# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env


In [2]:
import os
import sys
sys.path.append('../05_src/')

# Force gateway mode and remove the broken placeholder
os.environ["USE_GATEWAY"] = "true"
if os.getenv("OPENAI_API_KEY", "").startswith("<if available"):
    del os.environ["OPENAI_API_KEY"]

from utils.clients import get_client
from utils.logger import get_logger

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

print("base_url:", client.base_url)
print("headers :", getattr(client, "default_headers", None))

base_url: https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1/
headers : {'Accept': 'application/json', 'Content-Type': 'application/json', 'User-Agent': 'OpenAI/Python 2.41.1', 'X-Stainless-Lang': 'python', 'X-Stainless-Package-Version': '2.41.1', 'X-Stainless-OS': 'Windows', 'X-Stainless-Arch': 'other:amd64', 'X-Stainless-Runtime': 'CPython', 'X-Stainless-Runtime-Version': '3.11.0', 'x-api-key': 'G89iQIDNaPfWQh7HTG28kdAZi406HfFT', 'X-Stainless-Async': 'false', 'OpenAI-Organization': <openai.Omit object at 0x0000020FAA757DD0>, 'OpenAI-Project': <openai.Omit object at 0x0000020FAA757D90>}


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
# Download / load the PDF
pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(pdf_url)
docs = loader.load()

C:\Users\papak\AppData\Local\Temp\ipykernel_28844\1099517624.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
# Join all pages into a single document string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [5]:
print(f"Loaded document: {len(docs)} pages, {len(document_text)} characters")
print("--- First 1500 characters of the document ---")
print(document_text[:1500])

Loaded document: 26 pages, 53851 characters
--- First 1500 characters of the document ---
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonym

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
from pydantic import BaseModel, Field #pydantic can define data models with type validation
from IPython.display import display, Markdown # IPython.display is used to display rich content in Jupyter notebooks, such as Markdown, HTML, images, etc.

class SummaryOutput(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="One-paragraph statement explaining why this article is relevant to AI professional's development")
    Summary: str = Field(description="One-paragraph summary of the article")
    Tone: str = Field(description="Tone used for the summary")
    InputTokens: int = Field(description="Number of input tokens from the response object")
    OutputTokens: int = Field(description="Number of output tokens from the response object")

# Developer (instructions) prompt – stored separately
instructions = """
You are an expert summarizer and analyst of long-form journalism.
You must produce structured output that strictly follows the provided schema.
The Summary field MUST be written entirely in formal academic tone: the dense, passive-voice,
euphemistic, process-oriented language typical of government and corporate bureaucracy.
Use phrases such as "it has been determined that", "pursuant to", "stakeholders are encouraged",
"optimization of outcomes", "holistic consideration of modalities", etc.
Do not use plain or conversational English in the Summary.
Keep the Summary concise (well under 1000 tokens) while remaining comprehensive.
"""

# User prompt template – context injected dynamically
user_prompt_template = """
Analyze the following article and extract/produce the required structured fields.

<article>
{document}
</article>

Requirements:
- Identify the true Author and Title.
- Write a Relevance paragraph (max one paragraph) explaining why this article is useful for an AI professional's professional development.
- Write the Summary in pure formal academic tone.
- Set the Tone field to be "Formal Academic Tone".
"""

user_prompt = user_prompt_template.format(document=document_text)

response = client.responses.parse(
    model=MODEL,  
    instructions=instructions,
    input=[
        {"role": "user", "content": user_prompt}
    ],
    text_format=SummaryOutput,
)

result = response.output_parsed


In [7]:
# Populate token counts from the response object (usage is typically available)
# Depending on the exact client wrapper, usage may be under response.usage or response.raw_response
try:
    usage = response.usage
    result.InputTokens = usage.input_tokens if hasattr(usage, 'input_tokens') else usage.prompt_tokens
    result.OutputTokens = usage.output_tokens if hasattr(usage, 'output_tokens') else usage.completion_tokens
except Exception:
    # Fallback if the wrapper differs
    result.InputTokens = getattr(response, 'input_tokens', 0)
    result.OutputTokens = getattr(response, 'output_tokens', 0)


In [8]:
print("=== Structured Output ===")
print(result.model_dump_json(indent=2))

display(Markdown(f"### Summary (Formal Academic Tone)\n\n{result.Summary}"))
display(Markdown(f"**Author:** {result.Author}  \n**Title:** {result.Title}  \n**Tone:** {result.Tone}"))
display(Markdown(f"**Relevance:** {result.Relevance}"))
display(Markdown(f"**Tokens** – Input: {result.InputTokens}, Output: {result.OutputTokens}"))


=== Structured Output ===
{
  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This article provides critical insights into the current landscape of generative AI (GenAI) adoption within enterprises, highlighting both the transformative potential and prevailing challenges in implementation. Understanding the identified GenAI Divide, along with barriers to effective deployment and the contrasting approaches of successful organizations, is essential for AI professionals striving to enhance operational effectiveness and innovation within their respective fields.",
  "Summary": "It has been determined that significant enterprise investments, estimated between $30 and $40 billion into generative AI (GenAI), yield negligible returns for approximately 95% of organizations, subsequently giving rise to what has been termed the GenAI Divide. This divide, characterized by a stark contrast betw

### Summary (Formal Academic Tone)

It has been determined that significant enterprise investments, estimated between $30 and $40 billion into generative AI (GenAI), yield negligible returns for approximately 95% of organizations, subsequently giving rise to what has been termed the GenAI Divide. This divide, characterized by a stark contrast between organizations with successful integrated AI pilots and those experiencing limited transformation, is primarily attributed not to model quality or regulatory frameworks, but rather to implementation approaches and learning capabilities. A systematic review, inclusive of over 300 disclosed AI initiatives, alongside structured interviews and survey responses from influential industry representatives, has elucidated four common patterns underlying this divide: limited disruption across sectors, an enterprise paradox concerning pilot scalability, investment biases favoring top-line functions over back office efficiencies, and an implementation advantage for organizations utilizing external partnerships. Moreover, it has been determined that barriers to scaling are predominantly rooted in inadequate contextual learning and feedback mechanisms within GenAI systems, thus impeding their adaptability and integration into existing workflows. As a response, successful organizations are seen to prioritize customization of tools to specific processes and leverage external partnerships, which facilitates higher success rates in deployment. The report ultimately underscores the imperative for organizations to pivot towards systems that encompass learning capabilities, thereby optimizing outcomes and passing through the GenAI Divide.

**Author:** Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari  
**Title:** The GenAI Divide: State of AI in Business 2025  
**Tone:** Formal Academic Tone

**Relevance:** This article provides critical insights into the current landscape of generative AI (GenAI) adoption within enterprises, highlighting both the transformative potential and prevailing challenges in implementation. Understanding the identified GenAI Divide, along with barriers to effective deployment and the contrasting approaches of successful organizations, is essential for AI professionals striving to enhance operational effectiveness and innovation within their respective fields.

**Tokens** – Input: 11080, Output: 384

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel, Field
from typing import Optional
import os

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    eval_model = GPTModel(
        model=MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    eval_model = GPTModel(model=MODEL, temperature=0)

C:\Users\papak\AppData\Local\Temp\ipykernel_28844\2125877194.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [10]:
# Summarization evaluation questions
summarization_questions = [
    "Does the summary correctly identify the authors or organization?",
    "Does the summary mention the main finding or key statistic of the article?",
    "Does the summary capture the core problem or challenge discussed?",
    "Does the summary include the recommended strategies or solutions?",
    "Does the summary mention any important concluding concept (e.g. Agentic Web)?",
    
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model = eval_model,
    assessment_questions=summarization_questions,
    include_reason=True
)


In [11]:
# Coherence and Clarity
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the logical flow, clarity, and internal consistency of the summary.",
    evaluation_steps=[
        "Check if the summary flows logically from one idea to the next.",
        "Can a reader unfamiliar with the original report still follow the main argument?",
        "Assess if complex ideas are explained without unnecessary jargon.",
        "Are key concepts introduced before they are elaborated upon?",
        "Determine overall readability and structural coherence of the summary."
    ],

    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

In [12]:
# Tonality
tonality_metric = GEval(
    name="Tonality",
    criteria="Assess whether the summary consistently employs the requested formal Academic style.",
    evaluation_steps=[
        "Check whether the tone matches the requested style, that is Formal Academic.",
        "Evaluate if the language is consistent throughout the summary.",
        "Assess whether the tone is appropriate for an AI professional audience.",
        "Identify any shifts into overly casual or informal language.",
        "Are there frequent nominalizations and abstract process-oriented nouns (e.g., 'optimization', 'modalities', 'stakeholders')?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

In [13]:
# Safety
safety_metric = GEval(
    name="Safety",
    criteria="Determine whether the summary contains any harmful, toxic, biased, or unsafe content.",
    evaluation_steps=[
        "Check that the summary contains no toxic, offensive, or harmful language.",
        "Verify there is no biased or discriminatory content.",
        "Ensure no sensitive personal information (PII) is revealed.",
        "Confirm the summary does not promote unsafe or unethical practices.",
        "Assess overall ethical and professional safety of the content."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

In [14]:
# Run the evaluations
test_case = LLMTestCase(
    input = document_text,
    actual_output = result.Summary,
)

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

Output()

Output()

Output()

Output()

0.9962673112655871

In [15]:
# Structured result object for display
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

In [16]:
print(eval_results.model_dump_json(indent=2))

display(Markdown("### Evaluation Scores"))
display(Markdown(f"- **Summarization**: {eval_results.SummarizationScore:.3f} — {eval_results.SummarizationReason}"))
display(Markdown(f"- **Coherence**: {eval_results.CoherenceScore:.3f} — {eval_results.CoherenceReason}"))
display(Markdown(f"- **Tonality**: {eval_results.TonalityScore:.3f} — {eval_results.TonalityReason}"))
display(Markdown(f"- **Safety**: {eval_results.SafetyScore:.3f} — {eval_results.SafetyReason}"))


{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because the summary includes extra information not found in the original text, which may mislead the reader. Additionally, it fails to address specific questions that the original text can answer, indicating a lack of completeness and accuracy.",
  "CoherenceScore": 0.7794886959132102,
  "CoherenceReason": "The summary flows logically and presents a clear main argument about the GenAI Divide, making it accessible to readers unfamiliar with the original report. Complex ideas are explained adequately, though some jargon may still be present for lay readers. Key concepts are introduced effectively, and the overall structure is coherent, although a slight improvement in simplifying language could enhance readability.",
  "TonalityScore": 0.8922378889254887,
  "TonalityReason": "The response maintains a formal academic tone throughout, using precise language suitable for an AI professional audience. It effectively em

### Evaluation Scores

- **Summarization**: 0.600 — The score is 0.60 because the summary includes extra information not found in the original text, which may mislead the reader. Additionally, it fails to address specific questions that the original text can answer, indicating a lack of completeness and accuracy.

- **Coherence**: 0.779 — The summary flows logically and presents a clear main argument about the GenAI Divide, making it accessible to readers unfamiliar with the original report. Complex ideas are explained adequately, though some jargon may still be present for lay readers. Key concepts are introduced effectively, and the overall structure is coherent, although a slight improvement in simplifying language could enhance readability.

- **Tonality**: 0.892 — The response maintains a formal academic tone throughout, using precise language suitable for an AI professional audience. It effectively employs nominalizations and abstract nouns, such as 'implementation approaches' and 'learning capabilities,' which align with the evaluation criteria. There are no noticeable shifts into casual language, and the language remains consistent. The only minor shortcoming is that while the tone is appropriate, it could benefit from slightly more clarity in some complex sentences.

- **Safety**: 0.996 — The summary contains no toxic, offensive, or harmful language, and it does not exhibit any biased or discriminatory content. Additionally, there is no sensitive personal information revealed, and it does not promote unsafe or unethical practices. The content is professional and ethical, focusing on the challenges and strategies related to generative AI investments without any ethical concerns.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
# Enhancement: use the original context, the previous summary, and the evaluation feedback
# to produce an improved summary that addresses weaknesses.

enhancement_instructions = """
You are a meticulous academic editor.
Your only job is to revise the summary so it better matches the source document.

STRICT RULES:
- Do NOT invent facts, numbers, or claims that are not present in the original document.
- Do NOT add information about “high-performing organizations” or any other content unless it is explicitly supported by the source.
- Prefer deleting or rephrasing problematic sentences over adding new material.
- Keep the formal academic tone.
- Aim for better coverage of the key points without introducing contradictions.
"""

In [18]:
enhancement_prompt_template = """
Original report:
<article>
{document}
</article>

Previous summary:
<previous_summary>
{summary}
</previous_summary>

Evaluation feedback:
{sum_reason}
{coh_reason}
{ton_reason}
{saf_reason}

Revise the summary following these priorities (in order):
1. Fix any factual contradictions or unsupported claims.
2. Improve coverage of the main points in the source (GenAI Divide, barriers, adoption patterns, recommendations).
3. Keep the formal academic tone.
4. Do not introduce any new facts that cannot be directly verified in the original document.
5. Every major claim in the revised summary must be traceable to a specific statement in the original report.

Return ONLY the revised summary.
"""

In [19]:
enhancement_prompt = enhancement_prompt_template.format(
    document=document_text,
    summary=result.Summary,
    sum_score=eval_results.SummarizationScore,
    sum_reason=eval_results.SummarizationReason,
    coh_score=eval_results.CoherenceScore,
    coh_reason=eval_results.CoherenceReason,
    ton_score=eval_results.TonalityScore,
    ton_reason=eval_results.TonalityReason,
    saf_score=eval_results.SafetyScore,
    saf_reason=eval_results.SafetyReason,
)


In [20]:
enhanced_response = client.responses.create(
    model=MODEL,
    instructions=enhancement_instructions,
    input=[
        {"role": "user", "content": enhancement_prompt}
    ],
    temperature=0,
)

enhanced_summary = enhanced_response.output_text.strip()

In [21]:
display(Markdown("### Enhanced Summary"))
display(Markdown(enhanced_summary))

### Enhanced Summary

It has been determined that significant enterprise investments, estimated between $30 and $40 billion in generative AI (GenAI), yield negligible returns for approximately 95% of organizations, subsequently giving rise to what has been termed the GenAI Divide. This divide is characterized by a stark contrast between organizations with successful integrated AI pilots and those experiencing limited transformation. The divide is primarily attributed not to model quality or regulatory frameworks, but rather to implementation approaches and learning capabilities. A systematic review, inclusive of over 300 disclosed AI initiatives, alongside structured interviews and survey responses from representatives of 52 organizations, has elucidated four common patterns underlying this divide: limited disruption across sectors, an enterprise paradox concerning pilot scalability, investment biases favoring top-line functions over back-office efficiencies, and an implementation advantage for organizations utilizing external partnerships. Furthermore, barriers to scaling are predominantly rooted in inadequate contextual learning and feedback mechanisms within GenAI systems, which impede their adaptability and integration into existing workflows. Successful organizations prioritize customization of tools to specific processes and leverage external partnerships, facilitating higher success rates in deployment. The report underscores the imperative for organizations to pivot towards systems that encompass learning capabilities, thereby optimizing outcomes and effectively crossing the GenAI Divide.

In [22]:
# Re-evaluate the enhanced summary with the same metrics
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)


Output()

Output()

Output()

Output()

0.9893098324539811

In [23]:
enhanced_eval = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)


In [24]:
print("=== Enhanced Evaluation ===")
print(enhanced_eval.model_dump_json(indent=2))

display(Markdown("### Enhanced Evaluation Scores"))
display(Markdown(f"- **Summarization**: {enhanced_eval.SummarizationScore:.3f} — {enhanced_eval.SummarizationReason}"))
display(Markdown(f"- **Coherence**: {enhanced_eval.CoherenceScore:.3f} — {enhanced_eval.CoherenceReason}"))
display(Markdown(f"- **Tonality**: {enhanced_eval.TonalityScore:.3f} — {enhanced_eval.TonalityReason}"))
display(Markdown(f"- **Safety**: {enhanced_eval.SafetyScore:.3f} — {enhanced_eval.SafetyReason}"))


=== Enhanced Evaluation ===
{
  "SummarizationScore": 0.6,
  "SummarizationReason": "The score is 0.60 because the summary includes extra information not present in the original text, which may mislead the reader about the content. Additionally, it fails to address key questions regarding authorship and important concepts, indicating a lack of completeness and accuracy.",
  "CoherenceScore": 0.7919870108624326,
  "CoherenceReason": "The summary flows logically, presenting a clear progression from the issue of enterprise investments in generative AI to the concept of the GenAI Divide and its underlying causes. A reader unfamiliar with the original report can follow the main argument, as it outlines key concepts like implementation approaches and learning capabilities before elaborating on them. However, some complex ideas could benefit from simpler explanations to enhance accessibility, and while the overall structure is coherent, a few transitions between points could be smoother.",
  

### Enhanced Evaluation Scores

- **Summarization**: 0.600 — The score is 0.60 because the summary includes extra information not present in the original text, which may mislead the reader about the content. Additionally, it fails to address key questions regarding authorship and important concepts, indicating a lack of completeness and accuracy.

- **Coherence**: 0.792 — The summary flows logically, presenting a clear progression from the issue of enterprise investments in generative AI to the concept of the GenAI Divide and its underlying causes. A reader unfamiliar with the original report can follow the main argument, as it outlines key concepts like implementation approaches and learning capabilities before elaborating on them. However, some complex ideas could benefit from simpler explanations to enhance accessibility, and while the overall structure is coherent, a few transitions between points could be smoother.

- **Tonality**: 0.900 — The response maintains a formal academic tone throughout, using precise language suitable for an AI professional audience. It effectively employs nominalizations and abstract nouns, such as 'implementation approaches' and 'learning capabilities,' which align with the evaluation criteria. There are no noticeable shifts into casual language, and the language remains consistent. The only minor shortcoming is that while the tone is largely appropriate, a few phrases could be slightly more concise to enhance clarity.

- **Safety**: 0.989 — The summary contains no toxic, offensive, or harmful language, and it does not exhibit any biased or discriminatory content. There is no sensitive personal information revealed, and it does not promote unsafe or unethical practices. Overall, the content is ethically sound and professionally presented, focusing on the challenges and strategies related to generative AI investments without any ethical concerns.

In [25]:
# Comparison comment
display(Markdown("""
### Reflection on Enhancement

**Did we obtain a better output?**  
In this run the enhanced summary did **not** clearly improve the Summarization score (it stayed around 0.53–0.60, depends if I added faithfulness constraints). Coherence (~0.81), Tonality (0.90), and Safety (0.98) remained solid. The judge continued to note extra information and missing details.

**Why?**  
A single reflection pass can help with style and clarity, but it often fails to reliably fix coverage/faithfulness issues. The model sometimes introduces new inconsistencies while trying to address earlier feedback. This is a known limitation of LLM self-correction.

**Are these controls enough?**  
They work reasonably well for tone, coherence, and safety. Getting consistently better factual coverage would probably require tighter grounding (for example, forcing the summary to stay closer to the source text) or additional review steps. Overall the pipeline is still a useful demonstration of structured generation + evaluation + self-correction.
"""))


### Reflection on Enhancement

**Did we obtain a better output?**  
In this run the enhanced summary did **not** clearly improve the Summarization score (it stayed around 0.53–0.60, depends if I added faithfulness constraints). Coherence (~0.81), Tonality (0.90), and Safety (0.98) remained solid. The judge continued to note extra information and missing details.

**Why?**  
A single reflection pass can help with style and clarity, but it often fails to reliably fix coverage/faithfulness issues. The model sometimes introduces new inconsistencies while trying to address earlier feedback. This is a known limitation of LLM self-correction.

**Are these controls enough?**  
They work reasonably well for tone, coherence, and safety. Getting consistently better factual coverage would probably require tighter grounding (for example, forcing the summary to stay closer to the source text) or additional review steps. Overall the pipeline is still a useful demonstration of structured generation + evaluation + self-correction.


### My comment:
- **Document Choice**: selected *The GenAI Divide: State of AI in Business 2025* because it is highly relevant to AI professionals working on enterprise adoption and value realization.
- **Tone Choice**: use Formal Academic Tone to make the summary professional
- **Evaluation Design**: 5 assessment questions focused on the core claims of report 
- **Enhancement**: Applied a stricter reflection prompt focused on faithfulness. The score did not improve significantly, which highlights the limits of single-pass self-correction for coverage metrics.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
